# Retrieval de teses e dissertações que fazem uso de dados públicos

In [93]:
import json
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
import re

import xavy.dataframes as xd
import xavy.explore as xe

## Funções

In [32]:
def fillnone(entry, replacement):
    """
    Return `entry` if it is not None. Otherwise, return `replacement`.
    """
    if entry == None:
        return replacement
    else:
        return entry


def extract(usecases, key):
    """
    Given a list of dicts `usecases`, return a list of all properties 
    of the dicts stored under `key`.
    """
    values = [uc[key] for uc in usecases]
    return values


def select(usecases, key, value, exact=True):
    """
    Return a subset of elements in list of dict `usecases` where 
    property stored under `key` has `value`. If `exact` is False,
    select elements whose value (str) has a substring given by
    `value`.
    """
    if exact == True:
        sel = list(filter(lambda uc: uc[key] == value, usecases))
    else:
        sel = list(filter(lambda uc: fillnone(uc[key], '').find(value) != -1, usecases))
    return sel


class translate_dict(dict):
    """
    A dict that returns the key used if no translation was provided for it.
    """
    def __missing__(self,key):
        return key


def read_jsonl(path):
    """
    Read a JSONL file and return a list of dictionaries.

    Parameters
    ----------
    path : str or Path
        Path to the JSONL file.

    Returns
    -------
    list[dict]
        One dictionary per line in the file.
    """
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue  # skip empty lines
            try:
                #records.append(line)
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_num}") from e
    return records


def extract_xml_tag(text: str, tag: str):
    """
    Extract the content of the first XML tag from a string.

    Parameters
    ----------
    text : str
        String containing XML.
    tag : str
        Tag name to extract.

    Returns
    -------
    str | None
        Inner text of the tag, or None if not found.
    """
    try:
        # Wrap text in a root tag to ensure valid XML
        wrapped = f"<root>{text}</root>"
        root = ET.fromstring(wrapped)

        elem = root.find(tag)
        return elem.text if elem is not None else None

    except ET.ParseError:
        return None


## Listagem de exemplos de organizações que publicam dados públicos

### Carrega dados

In [3]:
# Lê os dados:
with open('../dados/backups/usecases_bkp_2025-12-12.json', 'r') as f:
    data = json.load(f)
usecases_raw = data['data']

In [4]:
# Seleciona casos considerados de uso de dados públicos:
usecases = list(filter(lambda uc: (uc['status_published'] == True) or ((fillnone(uc['comment'], '').find('Ocultado porque') == -1) and (fillnone(uc['comment'], '').find('não atendem aos nossos critérios') == -1)) , usecases_raw))

# Seleciona casos considerados como não consistentes com nosso critérios de coleta:
not_usecases = list(filter(lambda uc: not ((uc['status_published'] == True) or ((fillnone(uc['comment'], '').find('Ocultado porque') == -1) and (fillnone(uc['comment'], '').find('não atendem aos nossos critérios') == -1))) , usecases_raw))

In [5]:
# Cria uma tabela de datasets utilizados:
ds_list = pd.Series(extract(usecases, 'datasets')).explode().dropna().tolist()
ds_df = pd.DataFrame(ds_list)

### Lista de conjuntos de dados

In [6]:
examples_datasets = '; '.join(ds_df['data_name'].sample(10, random_state=179).unique())
examples_datasets

'Votações Nominais e Abertas; Dados Abertos do Governo Federal; Investimentos em Pesquisa - Doenças Tropiciais Negligenciadas e Malária; Dados de crédito de Pessoa Jurídica do Banco Central; Violência doméstica, sexual e/ou outras violências contra as mulheres; Resultados - 2002; Operador aeroportuário - Tarifas Aeroportuárias: Tetos Tarifários e Reajustes Tarifários; Densidade Demográfica de Alagoas; Taxa de alfabetização da população, total e quilombola, de 15 anos ou mais de idade, por sexo, grupos de idade, localização e situação do domicílio; Metadados da Base Siconv'

In [7]:
examples_data_providers = '; '.join(ds_df['data_institution'].sample(20, random_state=78453).unique())
examples_data_providers

'INEP; Ministério dos Direitos Humanos e da Cidadania; Instituto Internacional Arayara; Voos e operações aéreas - Registro de Serviços Aéreos;  Ministério do Desenvolvimento e Assistência Social, Família e Combate à Fome - MDS; Câmara Municipal de Feira de Santana; Banco Central do Brasil; IBGE; MapBiomas; Agência Nacional de Aviação Civil - ANAC; Câmara dos Deputados;  Agência Nacional do Petróleo, Gás Natural e Biocombustíveis - ANP; CAPES; Ministério dos Transportes; INPE; Ministério da Saúde; Instituto Brasileiro de Geografia e Estatística - IBGE'

## Carrega os dados de teses e dissertações

**FAZER**
* Remover duplicados
* Colocar identificador único e passar aos processos do GPT como `custom_id` (talvez usar uri).
* Entender por que algumas teses aparecem mais de uma vez com URIs diferentes e pequenas alterações em algum campo.

In [3]:
# Load data:
teses_df = pd.read_csv('../dados/dspaces/tccs-dissertacoes-teses-2024-ufpe.csv')
# Clean abstract:
teses_df['resumo'] = teses_df['resumo'].str.replace('\r\n', ' ').fillna('')
# Export to other formats:
text_data = teses_df['titulo'] + '\n\n' + teses_df['resumo'] + '\n\n' + teses_df['palavras_chave']
teses_records = teses_df[['titulo', 'resumo', 'palavras_chave']].to_dict(orient='records')

## Avaliação de resumos de trabalhos acadêmicos com GPT

In [9]:
from openai import OpenAI

from xavy.utils import load_env_vars

In [95]:
class PublicDataUsageDetector:
    """
    An academic work inspector that uses OpenAI's LLMs to classify 
    works into "Uses public data" or "Does not use public data".
    """
    def __init__(self, model_name, key_path, prompt_template, examples_datasets, examples_data_providers, temperature=1):
        """
        Parameters
        ----------
        model_name : str
            Name of the OpenAI model to use (e.g., 'gpt-5-nano').
        key_path : str
            Path to a file containing environment variables, including
            the OpenAI API key.
        prompt_template : str
            Path to a text file containing the prompt template used
            for classification.
        examples_datasets : str
            Example dataset names used to populate the prompt template
            and guide the model's classification.
        examples_data_providers : str
            Example data provider names used to populate the prompt
            template and guide the model's classification.
        temperature : float, optional
            Sampling temperature for the language model. Higher values
            increase randomness, while lower values make the output more
            deterministic. Default is 1.
        """
        # Save to internal attributes:
        self.model_name = model_name
        self.key_path = key_path
        self.prompt_template = prompt_template
        self.template_vars = {'examples_datasets': examples_datasets, 'examples_data_providers': examples_data_providers}
        self.temperature = temperature
        
        # Load the API key and other parameters:
        self.env = load_env_vars(key_path)
        self.env['OPENAI_PROJECT_ID'] = 'proj_ZwJObp10wyAmWtBSIESV70Lw'

        # Create a client:
        self.client = OpenAI(api_key=self.env['OPENAI_API_KEY'], project=self.env['OPENAI_PROJECT_ID'])

        # Read template:
        with open(prompt_template, 'r') as f:
            self.template = f.read()
        

    def build_prompt(self, data_dict: dict):
        """
        Return a string containing the prompt build for classifying 
        work described in `data_dict` (dict).

        `data_dict` should contain the keys: 'titulo', 'resumo' and 'palavras_chave'.
        """
        # Create prompt:
        prompt_input = self.template_vars.copy()
        prompt_input.update(data_dict)
        prompt = self.template % prompt_input
        return prompt

    
    def inspect(self, titulo, resumo, palavras_chave):
        """
        Call OpenAI's model to classify the work with the specified 
        metadata.
        """
        
        # Create prompt:
        data_dict = {'titulo': titulo, 'resumo': resumo, 'palavras_chave': palavras_chave}
        prompt = self.build_prompt(data_dict)
        
        # Ask LLM:
        completion = self.client.chat.completions.create(model=self.model_name, temperature=self.temperature, messages=[{"role": "developer", "content": prompt}])
        # Get response:
        answer = completion.choices[0].message.content
        
        return answer

    
    def build_batch_instance(self, custom_id: str, data_dict: dict, serialize=False, endpoint='/v1/chat/completions'):
        """
        Create the payload (dict) of a batch request for a single academic work.

        Parameters
        ----------
        custom_id : str
            An ID that represents the request.
        data_dict : dict
            It should contain the keys: 'titulo', 'resumo' and 'palavras_chave'.
        serialize : bool
            If True, return a string in JSON format. Otherwise, return a dict.
        endpoint : str
            Which OpenAI's endpoint to use for the request.
        """
        prompt = self.build_prompt(data_dict)
        body = {'model': self.model_name, 'messages': [{'role': 'developer', 'content': prompt}]}
        api_request = {'custom_id': custom_id, 'method': 'POST', 'url': endpoint, 'body': body}
        if serialize == False:
            return api_request
        else:
            return json.dumps(api_request, ensure_ascii=False)

    
    def build_batch(self, data_records: list, save_to=None, id_prefix='request-', id_offset=0, endpoint='/v1/chat/completions'):
        """
        Create a string in JSONL format in which each line is a JSON 
        specifying the request for OpenAI's LLM model.

        Parameters
        ----------
        data_records : list of dict 
            Metadata of the academic works to classify. Each entry 
            should contain the keys: 'titulo', 'resumo' and 
            'palavras_chave'.
        save_to : str, Path or None
            If `save_to` (str | Path) is provided, save the JSONL to 
            the specified file and return nothing.
        id_prefix : str
            Prefix for the 'custom_id' of each request.
        id_offset : int
            The number associated to the first request in the batch.
        endpoint : str
            Which OpenAI's endpoint to use for the request.            
        """
        jsonl = '\n'.join([self.build_batch_instance(id_prefix + str(i + id_offset), d, serialize=True, endpoint=endpoint) for i, d in enumerate(data_records)])
        if save_to == None:
            return jsonl
        else: 
            with open(save_to, 'w') as f:
                f.write(jsonl)


    def upload_batch(self, data_records: list, save_to=None, id_prefix='request-', id_offset=0, endpoint='/v1/chat/completions'):
        """
        Upload a batch request for public data usage classification to 
        OpenAI's LLM model.

        Parameters
        ----------
        data_records : list of dict 
            Metadata of the academic works to classify. Each entry 
            should contain the keys: 'titulo', 'resumo' and 
            'palavras_chave'.
        save_to : str, Path or None
            If `save_to` (str | Path) is provided, save the JSONL to 
            the specified file and return nothing.
        id_prefix : str
            Prefix for the 'custom_id' of each request.
        id_offset : int
            The number associated to the first request in the batch.
        endpoint : str
            Which OpenAI's endpoint to use for the request.

        Returns
        -------
        batch_input_file : FileObject
            An OpenAI API object that describes a file uploaded to 
            OpenAI Platform. The file contain the information 
            required to run a batch.
        """            
        # Stream payload:
        if save_to == None:
            batch = self.build_batch(data_records, save_to=save_to, id_prefix=id_prefix, id_offset=id_offset, endpoint=endpoint).encode()
            batch_input_file = self.client.files.create(file=batch, purpose="batch")
        # Save a copy of payload:
        else:
            self.build_batch(data_records, save_to=save_to, id_prefix=id_prefix, id_offset=id_offset, endpoint=endpoint)
            batch_input_file = self.client.files.create(file=open(save_to, 'rb'), purpose="batch")

        return batch_input_file


    def run_batch(self, batch_file, batch_description='Public data usage detector', endpoint='/v1/chat/completions'):
        """
        Request to run the batch specified in a file previously uploaded 
        to OpenAI's platform.

        Parameters
        ----------
        batch_file : FileObject
            Object returned by the function `self.upload_batch()`.
        batch_description : str
            Description to be added to the batch's metadata.
        endpoint : str
            Which OpenAI's endpoint to use for the request.            

        Returns
        -------
        batch_obj : Batch
            An OpenAI API object that describes a batch submitted to the
            OpenAI Platform.
        """
        batch_obj = self.client.batches.create(input_file_id=batch_file.id, endpoint=endpoint, completion_window='24h', metadata={"description": batch_description})

        return batch_obj


    def inspect_batch(self, data_records: list, save_to=None, id_prefix='request-', id_offset=0, batch_description='Public data usage detector', endpoint='/v1/chat/completions'):
        """
        Schedule the classification of academic works as "Uses public data" 
        or "Does not use public data" based on the provided metadata using 
        OpenAI's batch mode.

        Parameters
        ----------
        data_records : list of dict 
            Metadata of the academic works to classify. Each entry 
            should contain the keys: 'titulo', 'resumo' and 
            'palavras_chave'.
        save_to : str, Path or None
            If `save_to` (str | Path) is provided, save the JSONL to 
            the specified file.
        id_prefix : str
            Prefix for the 'custom_id' of each request.
        id_offset : int
            The number associated to the first request in the batch.
        batch_description : str
            Description to be added to the batch's metadata.
        endpoint : str
            Which OpenAI's endpoint to use for the request.            

        Returns
        -------
        batch_obj : Batch
            An OpenAI API object that describes a batch submitted to the
            OpenAI Platform. It can be used to monitor the batch's status.
        """        
        batch_input_file = self.upload_batch(data_records, save_to=save_to, id_prefix=id_prefix, id_offset=id_offset, endpoint=endpoint)
        batch_obj = self.run_batch(batch_input_file, batch_description=batch_description, endpoint=endpoint)
        return batch_obj


    def estimate_instance_tokens(self, batch_instance):
        """
        Estimate the number of tokens in the prompt of a batch
        instance.
    
        Parameters
        ----------
        batch_instance : dict
            Dict containing all the information required for running 
            OpenAI's model on an instance work in the batch 
            configuration.
    
        Returns
        -------
        tot_tokens : int
            Number of tokens estimated from the word counts in the
            prompt.
        """
        
        text_data = ' '.join(extract(batch_instance['body']['messages'], 'content'))
        n_words = np.array([len(t.split()) for t in text_data])
        tot_words = n_words.sum()
        tot_tokens = int(tot_words / 3481 * 5069) # Based on from https://platform.openai.com/tokenizer
        
        return tot_tokens


    def estimate_batch_tokens(self, data_records: dict):
        """
        Estimate the total number of input tokens in a batch.

        Parameters
        ----------
        data_records : list of dict
            Metadata of the academic works to classify. Each entry 
            should contain the keys: 'titulo', 'resumo' and 
            'palavras_chave'.

        Returns
        -------
        batch_tokens : int
            Estimate of the total number of tokens in the prompts
            used to classify all records in the `data_records` 
            list.
        """
        tokens = [self.estimate_instance_tokens(self.build_batch_instance(str(i), d)) for i, d in enumerate(data_records)]
        tokens = [self.build_batch_instance(str(i), d) for i, d in enumerate(data_records)]
        #batch_tokens = np.sum(tokens)
        return tokens
        return int(batch_tokens)

In [97]:
detector.estimate_batch_tokens(teses_records[:2])

[{'custom_id': '0',
  'method': 'POST',
  'url': '/v1/chat/completions',
  'body': {'model': 'gpt-5-nano',
   'messages': [{'role': 'developer',
     'content': '# ROLE\nYou are a data scientist expert in research metadata analysis and open data.\n\n# TASK\nYour task is to determine whether the Brazilian academic work described below makes use of publicly available datasets in its analysis.\n\n## DEFINITION\nFor this task, consider that a work "uses publicly available datasets" if it analyzes data that can be accessed online by the public without special authorization, such as:\n- Government or official statistics (e.g., IBGE, INEP, World Bank, WHO, and DATASUS);\n- Open government data portals (e.g., Portal Brasileiro de Dados Abertos, Geosampa);\n- Public administrative records;\n- Datasets made available on the Web by NGOs (e.g., Wikidata);\n- Open scientific, academic or environmental datasets;\n- Open satellite or geospatial data (e.g., Prodes, Landsat, Sentinel, OpenStreetMap).\n

In [96]:
# Instantiate classifier:
detector = PublicDataUsageDetector('gpt-5-nano', '../keys/openai.txt', 'open-data_llm-template.md', examples_datasets, examples_data_providers)

In [25]:
ini = 5
n_instances = 995
end = ini + n_instances
batch_file = f'gpt-batch_ufpe-2024_{ini}-{end}.jsonl'
batch_obj = detector.inspect_batch(teses_records[ini:end], save_to=batch_file, id_offset=ini, batch_description='Detection of public data usage on 2024 UFPE academic works (partition).')

In [27]:
batch = detector.client.batches.retrieve(batch_obj.id)
print(batch)

Batch(id='batch_694687f6f2188190b4c2dc0caec8fd6d', completion_window='24h', created_at=1766230006, endpoint='/v1/chat/completions', input_file_id='file-S1BZSoCo8xpwunSy1JDXUk', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1766231524, error_file_id=None, errors=None, expired_at=None, expires_at=1766316406, failed_at=None, finalizing_at=1766231467, in_progress_at=1766230069, metadata={'description': 'Detection of public data usage on 2024 UFPE academic works (partition).'}, output_file_id='file-TRJc1a8Krxi1DdeTE6rmz5', request_counts=BatchRequestCounts(completed=995, failed=0, total=995), model='gpt-5-nano-2025-08-07', usage={'input_tokens': 1255007, 'output_tokens': 337690, 'total_tokens': 1592697, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens_details': {'reasoning_tokens': 324032}})


In [28]:
data_records = teses_records[ini:end]


In [65]:
batch_instance = detector.build_batch_instance('teste', data_records[48])

'# ROLE\nYou are a data scientist expert in research metadata analysis and open data.\n\n# TASK\nYour task is to determine whether the Brazilian academic work described below makes use of publicly available datasets in its analysis.\n\n## DEFINITION\nFor this task, consider that a work "uses publicly available datasets" if it analyzes data that can be accessed online by the public without special authorization, such as:\n- Government or official statistics (e.g., IBGE, INEP, World Bank, WHO, and DATASUS);\n- Open government data portals (e.g., Portal Brasileiro de Dados Abertos, Geosampa);\n- Public administrative records;\n- Datasets made available on the Web by NGOs (e.g., Wikidata);\n- Open scientific, academic or environmental datasets;\n- Open satellite or geospatial data (e.g., Prodes, Landsat, Sentinel, OpenStreetMap).\n\nDo NOT consider as publicly available datasets:\n- Data collected exclusively via surveys, interviews, experiments, or fieldwork;\n- Proprietary, commercial, o

In [46]:
xe.checkMissing(teses_df[['titulo', 'resumo', 'palavras_chave']])

Colunas com valores faltantes:
   coluna    N     %
1  resumo  2.0  0.05


In [ ]:
xd.print_string_series(text_data.sample(10))

In [47]:
teses_df.loc[teses_df['resumo'].isnull()]

,autoria,orientacao,titulo,data_publicacao,fomento,publicador,campus,programa_curso,tipo,nivel_academico,total_paginas,resumo,idioma,uri,palavras_chave
1345,"Melo, Rosemary dos Santos Mendes de","Santos, Ana Lúcia Félix dos",GESTÃO ESCOLAR E INTELIGÊNCIA ARTIFICAL: O uso...,2024-03-25,NaN,Universidade Federal de Pernambuco,NaN,Programa de Pos Graduacao em Educacao Basica,Dissertação,NaN,NaN,NaN,por,https://repositorio.ufpe.br/handle/123456789/5...,Educação; Gestão Escolar; Tecnologias Digitais...
1504,"MUNIZ, DIRCIANE MARIA GONÇALVES COELHO","BONA, VIVIANE DE",A roda de conversa com foco na dimensão do sen...,2024-03-27,NaN,Universidade Federal de Pernambuco,NaN,Programa de Pos Graduacao em Educacao Basica,Dissertação,NaN,NaN,NaN,por,https://repositorio.ufpe.br/handle/123456789/5...,Roda de Conversa; Educação Emocional; Educação...


In [73]:
estimate_n_tokens(batch_instance)

7516

In [39]:
tot_words

313819

In [36]:
print(text_data[9])

PrevSífilis - Tecnologia educativa para prevenção da sífilis em adolescentes

Introdução: As infecções sexualmente transmissíveis acometem a população mundial e são consideradas um problema de saúde pública. Dentre as ferramentas que auxiliam na promoção da saúde, estão as diversas tecnologias educacionais, que tem como principal objetivo levar a educação em saúde para diferentes grupos. Objetivo: Descrever o processo de validação do jogo de tabuleiro “PrevSífilis” Metodologia: Trata-se de um estudo metodológico realizado em duas etapas: a primeira etapa consistiu no aperfeiçoamento do jogo e a segunda na validação do jogo com os juízes. A construção da tecnologia educativa Jogo de Tabuleiro “PrevSífilis” foi realizada em 2021 e o processo de aperfeiçoamento ocorreu em 2023. O jogo de tabuleiro aperfeiçoado foi apresentado a um grupo de juízes visando a sua validação. Para essa etapa de validação de conteúdo e aparência, os participantes foram juízes com experiência na área de IST/Sífi

In [30]:
extract(data_records, 'titulo')

['Taylor Swift\'s use of irony in "Blank Space" as a rhetorical device to respond to linguistic injury',
 'Autocratização comparada : democracias em retrocesso, eleições enviesadas e a expansão de autocracias eleitorais pelo mundo',
 'Uma ferramenta para a elaboração de feedbacks de atividades iniciais de programação integrada a um árbitro virtual',
 'Síntese de cristais líquidos a base de Bisperilenimidas e sua aplicação na obtenção de parâmetros anisotrópicos em RMN',
 'Aplicação das práticas de governança, gestão de risco e conformidade no serviço público, à luz do modelo TAM : um estudo no Instituto Federal da Bahia',
 'TRATAMENTO CONSERVADOR DE MÚLTIPLOS TUMORES ODONTOGÊNICOS EM PACIENTE INFANTIL: um relato de caso.',
 'Aplicação de metodologia de hypergrafos em dados da bolsa de valores',
 'Narrativas de agricultores familiares sobre o uso de agrotóxicos e suas consequências para a saúde no contexto sociopolítico da pandemia de Covid-19',
 'Validação de vídeo educacional para pre

In [ ]:
def estimate_tokens()

### Classify a single work on the fly

In [262]:
# Print work's metadata:
i = 4
print('%(titulo)s\n\n%(resumo)s\n\n%(palavras_chave)s' % teses_records[i])

Emergent high-order modularity in the human connectome network

The usage of graphs to model relationships between variables is a common and well- established practice, but, like every model, it has its limitations. In this work, we explore a workaround for one of those limitations: the correct representation of interactions between more than 2 variables. Specifically, we investigate the 3-by-3 interaction between areas of the brain, modeled using 3-uniform hypergraphs. Given that the pairwise network models of the brain exhibit a modular property, i.e., the presence of vertex modules in the graph, our goal here is to determine if the modular property persists in hypergraphs, using modularity as our metric with a threshold set at 0.3. We utilized data from rs-fMRI scans of 1018 young adults, provided by the Human Connectome Project (HCP). For each patient, we obtained 2 rs-fMRI scans. We observed the 3-by-3 interactions between areas of the brain using two information theory metrics: I

In [263]:
# Classify:
detector.inspect(**teses_records[i])

'Uses public data'

## Resultado da avaliação com GPT

In [119]:
# Load GPT results:
gpt_input  = read_jsonl('../dados/dspaces/tccs-dissertacoes-teses-2024-ufpe_gpt-in_0001-1000.jsonl')
gpt_output = read_jsonl('../dados/dspaces/tccs-dissertacoes-teses-2024-ufpe_gpt-out_0001-1000.jsonl')

In [123]:
# Create a GPT classification DataFrame:
gpt_df = pd.DataFrame()
#gpt_df['gpt_titulo'] = pd.Series([extract_xml_tag(g['body']['messages'][0]['content'], 'TITLE') for g in gpt_input])
gpt_df['gpt_titulo'] = pd.Series([re.findall('<TITLE>(.+)</TITLE>', g['body']['messages'][0]['content'])[0] for g in gpt_input])
gpt_df['gpt_id_in']  = pd.Series(extract(gpt_input, 'custom_id'))
gpt_df['gpt_id_out'] = pd.Series(extract(gpt_output, 'custom_id'))
gpt_df['class']      = pd.Series([g['response']['body']['choices'][0]['message']['content'] for g in gpt_output])

# TEMP: remove duplicates:
gpt_df.drop_duplicates(subset='gpt_titulo', inplace=True)

# Consistency check:
assert (gpt_df['gpt_id_in'] == gpt_df['gpt_id_out']).all()
# Integrity test:
assert xd.iskeyQ(gpt_df[['gpt_titulo']])

In [116]:
# Avaliações com títulos duplicados:
gpt_df.loc[gpt_df['gpt_titulo'].duplicated(keep=False)]

,gpt_titulo,gpt_id_in,gpt_id_out,class
252,Efeitos da criação extensiva de caprinos sobre...,request-252,request-252,Does not use public data
265,Efeitos da criação extensiva de caprinos sobre...,request-265,request-265,Does not use public data


In [136]:
# Teses com títulos duplicados:
duplicated_teses_df = teses_df.loc[teses_df['titulo'].duplicated(keep=False)]
xd.print_string_series(duplicated_teses_df.set_index('titulo')['uri'])

Efeitos da criação extensiva de caprinos sobre a comunidade de formigas e seus atributos funcionais morfológicos na Caatinga: https://repositorio.ufpe.br/handle/123456789/56378
Efeitos da criação extensiva de caprinos sobre a comunidade de formigas e seus atributos funcionais morfológicos na Caatinga: https://repositorio.ufpe.br/handle/123456789/58275
O que propôs Bolsonaro? Análise de conteúdo da agenda legislativa e administrativa (2019 - 2022): https://repositorio.ufpe.br/handle/123456789/56345
O que propôs Bolsonaro? Análise de conteúdo da agenda legislativa e administrativa (2019 - 2022): https://repositorio.ufpe.br/handle/123456789/56250
Galectina-9 como biomarcador de severidade em pacientes com COVID-19 leve e grave : implicações imunológicas e clínicas: https://repositorio.ufpe.br/handle/123456789/57770
Galectina-9 como biomarcador de severidade em pacientes com COVID-19 leve e grave : implicações imunológicas e clínicas: https://repositorio.ufpe.br/handle/123456789/57768
O ba

In [140]:
classified_teses_df = teses_df.drop_duplicates(subset=['titulo']).copy()
classified_teses_df = classified_teses_df.merge(gpt_df[['gpt_titulo', 'class']], left_on='titulo', right_on='gpt_titulo', how='outer')

In [147]:
missing_class = set(range(0, len(gpt_df))) - set(classified_teses_df.dropna(subset='class').index)
missing_class

set()

## Avaliação de resumos de trabalhos acadêmicos com BERT

## Avaliação de resumos de trabalhos acadêmicos com BERT

### Teste de modelos de IA

In [6]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

#### 1. bertimbau-large-InferBr-NLI

In [96]:
# 1. Load tokenizer and model from HuggingFace
tokenizer = AutoTokenizer.from_pretrained("felipesfpaula/bertimbau-large-InferBr-NLI")
model = AutoModelForSequenceClassification.from_pretrained("felipesfpaula/bertimbau-large-InferBr-NLI")

# 2. Encode a premise–hypothesis pair
premise = text_data.iloc[:2].tolist()
hypothesis = ['Este trabalho trata de violência urbana'] * 2
encoded = tokenizer(premise, hypothesis, return_tensors="pt", max_length=512, truncation=True, padding="max_length")

# 3. Run inference
with torch.no_grad():
    outputs = model(**encoded)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=1)
    answer = dict(zip(['Contradiction', 'Entailment', 'Neutral'], list(probs.T.tolist())))

answer

{'Contradiction': [6.719219527440146e-05, 0.0008191487286239862],
 'Entailment': [9.625975508242846e-05, 0.02135404385626316],
 'Neutral': [0.9998365640640259, 0.9778268337249756]}

#### 2. xlm_roberta_base_assin_fine_tuned 

In [42]:
from transformers import XLMRobertaTokenizer

In [43]:
model_path = "giotvr/portuguese-nli-3-labels"
premise = "As mudanças climáticas são uma ameaça séria para a biodiversidade do planeta."
hypothesis ="A biodiversidade do planeta é seriamente ameaçada pelas mudanças climáticas."
tokenizer = XLMRobertaTokenizer.from_pretrained(model_path, use_auth_token=True)
input_pair = tokenizer(premise, hypothesis, return_tensors="pt",padding=True, truncation=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, use_auth_token=True)

with torch.no_grad():
    logits = model(**input_pair).logits
probs = torch.nn.functional.softmax(logits, dim=-1)
probs, sorted_indices = torch.sort(probs, descending=True)
for i, score in enumerate(probs[0]):
    print(f"Class {sorted_indices[0][i]}: {score.item():.4f}")


/home/hxavier/ceweb/projetos/cordata/analises/env/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:2077: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/home/hxavier/ceweb/projetos/cordata/analises/env/lib/python3.8/site-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Class 2: 0.8273
Class 1: 0.1502
Class 0: 0.0226


In [60]:
model_path = "giotvr/portuguese-nli-3-labels"
premise = "As mudanças climáticas são uma ameaça séria para a biodiversidade do planeta."
hypothesis ="A biodiversidade do planeta é seriamente ameaçada pelas mudanças climáticas."
tokenizer = AutoTokenizer.from_pretrained(model_path, use_auth_token=True)
input_pair = tokenizer(premise, hypothesis, return_tensors="pt",padding=True, truncation=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, use_auth_token=True)

with torch.no_grad():
    logits = model(**input_pair).logits
probs = torch.nn.functional.softmax(logits, dim=-1)
probs, sorted_indices = torch.sort(probs, descending=True)
for i, score in enumerate(probs[0]):
    print(f"Class {sorted_indices[0][i]}: {score.item():.4f}")


/home/hxavier/ceweb/projetos/cordata/analises/env/lib/python3.8/site-packages/transformers/models/auto/tokenization_auto.py:809: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/home/hxavier/ceweb/projetos/cordata/analises/env/lib/python3.8/site-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Class 2: 0.8273
Class 1: 0.1502
Class 0: 0.0226


In [61]:
input_pair

{'input_ids': tensor([[     0,   1301, 119666,    501,   3141,  28447,      7,   3854,    788,
         165725,   6352,    399,    121,     10,      6, 179174,  16018,     54,
          33910,      5,      2,      2,     62,      6, 179174,  16018,     54,
          33910,    393,  17594,   1183, 165725,     85,  35335, 119666,    501,
           3141,  28447,      7,      5,      2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

#### 3. xlm-roberta-large-xnli

In [3]:
from transformers import pipeline

In [4]:
classifier = pipeline("zero-shot-classification",
                      model="joeddav/xlm-roberta-large-xnli")

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [46]:
# pose sequence as a NLI premise and label as a hypothesis
nli_model = AutoModelForSequenceClassification.from_pretrained('joeddav/xlm-roberta-large-xnli')
tokenizer = AutoTokenizer.from_pretrained('joeddav/xlm-roberta-large-xnli')

premise = text_data.iloc[1]
hypothesis = 'Um dos tópicos desse documento é política.'

# run through model pre-trained on MNLI
x = tokenizer.encode(premise, hypothesis, return_tensors='pt',
                     truncation_strategy='only_first')
logits = nli_model(x.to('cpu'))[0]

# we throw away "neutral" (dim 1) and take the probability of
# "entailment" (2) as the probability of the label being true 
entail_contradiction_logits = logits[:,[0,2]]
probs = entail_contradiction_logits.softmax(dim=1)
prob_label_is_true = probs[:,1]

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/home/hxavier/ceweb/projetos/cordata/analises/env/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:2869: FutureWarning: The `truncation_strategy` argument is deprecated and will be removed in a future version, use `truncation=True` to truncate examples to a max length. You c

In [16]:
logits[:, [0,2]].softmax(dim=1)

tensor([[0.0032, 0.9968]], grad_fn=<SoftmaxBackward0>)

In [14]:
logits[:, [0,2]]

tensor([[-3.4848,  0.5389,  2.2455]], grad_fn=<AddmmBackward0>)

In [41]:
# load model pretrained on MNLI
# pose sequence as a NLI premise and label (politics) as a hypothesis
premise = 'Who are you voting for president of the USA in 2020?'
hypothesis = 'This text is about sex.'
#antithesis = 'This text is not about sex.'

# run through model pre-trained on MNLI
input_ids = tokenizer.encode(premise, hypothesis, return_tensors='pt')
logits = nli_model(input_ids)[0]

# we throw away "neutral" (dim 1) and take the probability of
# "entailment" (2) as the probability of the label being true 
entail_contradiction_logits = logits[:,[0,2]]
probs = entail_contradiction_logits.softmax(dim=1)
true_prob = probs[:,1].item() * 100
#print(f'Probability that the label is true: {true_prob:0.2f}%')

nli_model(input_ids)[0].softmax(dim=1)

tensor([[0.9820, 0.0039, 0.0141]], grad_fn=<SoftmaxBackward0>)

In [59]:
input_ids

tensor([[    0, 40469,   621,   398, 20875,   214,   100, 13918,   111,    70,
          4602,    23, 11075,    32,     2,     2,  3293,  7986,    83,   959,
          1672,  1100,     5,     2]])

In [109]:
class NuclearNLI:

    def __init__(self, model_path):
        self.model_path = model_path
        self.model = AutoModelForSequenceClassification.from_pretrained(model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        if model_path == 'felipesfpaula/bertimbau-large-InferBr-NLI':
            self.max_length = 128
            self.labels = ['Contradiction', 'Entailment', 'Neutral']
        else:
            self.max_length = 512
            self.labels = ['Contradiction', 'Neutral', 'Entailment']

    def __call__(self, premise, hypothesis):
        input_pair = self.tokenizer(premise, hypothesis, return_tensors='pt', padding='max_length', truncation=True, max_length=self.max_length)
        with torch.no_grad():
            output = self.model(**input_pair)
            logits = output.logits
            probs  = logits.softmax(dim=1)
        return probs

   

In [124]:
#m = NuclearNLI('giotvr/portuguese-nli-3-labels', ['Contradiction', 'Neutral', 'Entailment'])
#m = NuclearNLI('joeddav/xlm-roberta-large-xnli', ['Contradiction', 'Neutral', 'Entailment'])
m = NuclearNLI('felipesfpaula/bertimbau-large-InferBr-NLI')

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("hapaxlegomenon/InferBR")
ds_train = ds['train']
ds_test  = ds['test']
ds_val   = ds['validation']

df_test = ds_test.to_pandas()
df_val  = ds_val.to_pandas()

In [136]:
pred_test = m(df_test['premise'].tolist(), df_test['hypothesis'].tolist()).argmax(dim=1)

In [150]:
pred_val = m(df_val['premise'].tolist(), df_val['hypothesis'].tolist()).argmax(dim=1)

In [143]:
pred_test

tensor([0, 2, 1,  ..., 2, 0, 1])

In [153]:
from sklearn.metrics import f1_score, precision_score

In [146]:
f1_score(df_test['label'], pred_test, average='macro')

0.9395691597747202

In [151]:
f1_score(df_val['label'], pred_val, average='macro')

0.9383888316788561

In [154]:
precision_score(df_test['label'], pred_test, average='macro')

0.9395668031793686

In [155]:
df_test['label'].value_counts()

label
0    573
1    568
2    564
Name: count, dtype: int64